In [6]:
# mass balance

import cobra
import pandas as pd

model = cobra.io.read_sbml_model("Helicobacter_pylori.xml")

from collections import defaultdict

imbalanced_reactions = []

for rxn in model.reactions:
    balance = defaultdict(float)
    
    for met, coeff in rxn.metabolites.items():

        if met.formula:
            try:
                elements = met.elements
                
                for element, count in elements.items():
                    balance[element] += coeff * count
            except Exception:
                pass

# remove very small numerical values
balance = {
    element: round(value, 6)
    for element, value in balance.items()
    if abs(value) > 1e-6
}

if balance:
    imbalanced_reactions.append({
        "Reaction" : rxn.id,
        "Name" : rxn.name,
        "Imbalance" : balance
    })

mass_balance_df = pd.DataFrame(imbalanced_reactions)

print("Total reactions : ", len(model.reactions))
print("Mass-imbalanced reactions : " , len(mass_balance_df))   

'' is not a valid SBML 'SId'.


Total reactions :  1025
Mass-imbalanced reactions :  1


In [8]:
print("Mass-imbalanced reactions : " , mass_balance_df)

Mass-imbalanced reactions :       Reaction        Name                                          Imbalance
0  biomass525  biomass525  {'C': -41.423097, 'H': -63.273871, 'N': -10.95...


In [10]:
mass_balance_df.head(20)

,Reaction,Name,Imbalance
0,biomass525,biomass525,"{'C': -41.423097, 'H': -63.273871, 'N': -10.95..."


In [12]:
# biomass rxn is imbalance

In [18]:
# charge balance
charge_imbalanced = []

for rxn in model.reactions:

    total_charge = 0

    for met, coeff in rxn.metabolites.items():

        if met.charge is not None:
            total_charge += coeff * met.charge

    total_charge = round(total_charge, 6)

    if abs(total_charge) > 1e-6:

        charge_imbalanced.append({
            "Reaction": rxn.id,
            "Name": rxn.name,
            "Charge_imbalance": total_charge
        })

charge_imbalance_df = pd.DataFrame(charge_imbalanced)

print("Total reactions:", len(model.reactions))
print("Charge-imbalanced reactions:", charge_imbalance_df)
print(len(charge_imbalance_df))

Total reactions: 1025
Charge-imbalanced reactions:          Reaction                                            Name  \
0          DM_btn                               Demand for biotin   
1     DM_thmpp(c)        demand reaction for Thiamine diphosphate   
2      EX_acac(e)                           Acetoacetate exchange   
3       EX_akg(e)                         2-Oxoglutarate exchange   
4    EX_alaasp(e)                   L-alanyl-L-aspartate exchange   
5    EX_alaglu(e)                   L-alanyl-L-glutamate exchange   
6     EX_arg_L(e)                             L-Arginine exchange   
7     EX_asp_L(e)                            L-Aspartate exchange   
8       EX_btn(e)                                 Biotin exchange   
9       EX_but(e)                      Butyrate (n-C4:0) exchange   
10      EX_ca2(e)                                Calcium exchange   
11      EX_cd2(e)                                Cadmium exchange   
12       EX_cl(e)                  exchange reaction

In [20]:
# internal rxns
internal_charge_imbalanced = charge_balance_df[
    ~charge_balance_df["Reaction"].str.startswith(("EX_", "DM_"))
]

print("Total charge-imbalanced:", len(charge_balance_df))
print("EX reactions:", charge_balance_df["Reaction"].str.startswith("EX_").sum())
print("DM reactions:", charge_balance_df["Reaction"].str.startswith("DM_").sum())
print("Other reactions:", len(internal_charge_imbalanced))

internal_charge_imbalanced

Total charge-imbalanced: 52
EX reactions: 49
DM reactions: 2
Other reactions: 1


,Reaction,Name,Charge_imbalance
51,biomass525,biomass525,1.116622
